# Chapitre 7 - Cours 4 : sérialisabilité, exécution en série, 2PL et SSI

Ce notebook couvre la section *Serializability* : *Actual Serial Execution*
(*Encapsulating transactions in stored procedures*, *Pros and cons of stored procedures*,
*Partitioning*, *Summary of serial execution*), *Two-Phase Locking (2PL)*
(*Implementation of two-phase locking*, *Performance of two-phase locking*, *Predicate
locks*, *Index-range locks*) et *Serializable Snapshot Isolation (SSI)* (*Pessimistic versus
optimistic concurrency control*, *Decisions based on an outdated premise*,
*Detecting stale MVCC reads*, *Detecting writes that affect prior reads*, *Performance of
serializable snapshot isolation*), ainsi que le résumé du chapitre.

**Plan.**

1. La sérialisabilité : définition et vérification (graphe de précédence).
2. Exécution réellement en série.
3. Procédures stockées et déterminisme.
4. Exécution en série et partitionnement.
5. Le verrouillage à deux phases (2PL).
6. Performance de 2PL.
7. Verrous de prédicat et verrous d'intervalle d'index.
8. L'isolation par instantané sérialisable (SSI).
9. Performance comparée : série, 2PL, SSI.

In [1]:
import itertools

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter

from ddia import transactions as tr
from ddia.transactions import (R, W, S, INC, run, plot_history, plot_precedence, precedence_graph,
                               is_serializable, simulate_load, LEVEL_LABELS)
from ddia.diagrams import Diagram
from ddia.timelines import SpaceTime
from ddia.plotting import (AMBER, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET,
                           annotate_bars, finish, set_style, subtitle)

set_style()
plt.rcParams["axes.titlepad"] = 16
# graduations des axes logarithmiques sans mathtext (la police du style n'a pas le signe moins)
LOGFMT = FuncFormatter(lambda v, _: f"{v:,.0f}".replace(",", " ") if v >= 1 else f"{v:g}")
pd.set_option("display.precision", 3)
rng = np.random.default_rng(0)